# Phase 2 Final QRC Encoding + Readout Probe

This is the final bounded optimization pass before freezing Phase 2.

It keeps the best leaky-input QRC architecture fixed and tests only:

1. clipped-linear angle encoding versus tanh-angle encoding;
2. small readout sensitivity around the best setting.

Base setting:

```text
PCA-6, 40-day lookback
leaky-integrated input, leak = 0.3
10 recent anchors
6-qubit full-topology TFIM-QRC
3 Trotter steps per anchor
3 virtual nodes per anchor
evolution_time = 0.5
ZXZZ observables
disorder_strength = 0.20
log target
```

This notebook deliberately avoids broad architecture search. If no clear gain appears, freeze the leaky03 result.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == 'notebooks':
    os.chdir('..')

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.evaluation.metrics import evaluate_volatility_forecast
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    _apply_single_qubit_gate,
    _fixed_disorder_factors,
    _safe_feature_target_correlations,
    diagnose_reservoir_feature_splits,
    evolve_tfim_step,
    initialize_zero_state,
    make_qrc_sequence_splits,
    observable_features,
    ry,
    select_anchor_indices,
)

In [ ]:
target = 'future_rv_20d'

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix='pca6',
)

raw_seq = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

print({k: (v[0].shape, v[1].shape) for k, v in raw_seq.items()})

In [ ]:
def leaky_integrate_window(window, leak=0.3):
    h = np.zeros(window.shape[1], dtype=float)
    out = np.zeros_like(window, dtype=float)
    for t, u_t in enumerate(window):
        h = (1.0 - leak) * h + leak * u_t
        out[t] = h
    return out


def leaky_integrate_windows(X, leak=0.3):
    return np.stack([leaky_integrate_window(window, leak) for window in X], axis=0)


leaky_seq = {
    split: (leaky_integrate_windows(X, leak=0.3), y, dates)
    for split, (X, y, dates) in raw_seq.items()
}

X_train, y_train, _ = leaky_seq['train']
X_val, y_val, _ = leaky_seq['val']
X_test, y_test, _ = leaky_seq['test']

## Custom encoder

The repository default uses clipped-linear angles. Here we add a local tanh-angle encoder without changing the module defaults.

In [ ]:
def encode_custom_angles(state, x, *, n_qubits, angle_max, encoding_mode='linear', tanh_scale=1.0):
    if encoding_mode == 'linear':
        angles = angle_max * (np.clip(x, -3.0, 3.0) / 3.0)
    elif encoding_mode == 'tanh':
        angles = angle_max * np.tanh(tanh_scale * x)
    else:
        raise ValueError(f'Unknown encoding_mode: {encoding_mode}')

    for j, theta in enumerate(angles):
        q = j % n_qubits
        state = _apply_single_qubit_gate(state, ry(theta), q, n_qubits)
    return state


def run_window_custom_encoder(window, config, *, encoding_mode='linear', tanh_scale=1.0):
    if window.ndim != 2:
        raise ValueError(f'Expected window shape (lookback, features), got {window.shape}')

    state = initialize_zero_state(config.qubits)
    anchor_indices = select_anchor_indices(config.lookback_days, config.anchor_count, config.anchor_policy)
    edge_factors, field_factors = _fixed_disorder_factors(config)

    anchor_features = []
    readout_steps = set(
        np.linspace(1, config.trotter_steps_per_anchor, config.virtual_nodes_per_anchor).round().astype(int)
    )

    for idx in anchor_indices:
        state = encode_custom_angles(
            state,
            window[idx],
            n_qubits=config.qubits,
            angle_max=config.angle_max,
            encoding_mode=encoding_mode,
            tanh_scale=tanh_scale,
        )
        for step in range(1, config.trotter_steps_per_anchor + 1):
            state = evolve_tfim_step(state, config, edge_factors=edge_factors, field_factors=field_factors)
            if config.collect_anchor_features and step in readout_steps:
                anchor_features.append(observable_features(state, config))

    if config.collect_anchor_features:
        return np.concatenate(anchor_features)
    return observable_features(state, config)


def build_features_custom_encoder(X, config, *, encoding_mode='linear', tanh_scale=1.0, verbose=False):
    rows = []
    for i, window in enumerate(X):
        if verbose and i % 250 == 0:
            print(f'QRC sample {i}/{len(X)}')
        rows.append(run_window_custom_encoder(window, config, encoding_mode=encoding_mode, tanh_scale=tanh_scale))
    return np.asarray(rows, dtype=float)

In [ ]:
base_config = TFIMQRCConfig(
    qubits=6,
    pca_components=6,
    lookback_days=40,
    anchor_count=10,
    anchor_policy='recent',
    observable_mode='zxzz',
    collect_anchor_features=True,
    topology='full',
    trotter_steps_per_anchor=3,
    virtual_nodes_per_anchor=3,
    coupling_scale=0.7,
    transverse_field=0.5,
    evolution_time=0.5,
    angle_max=np.pi / 2,
    ridge_alpha=3000.0,
    target_transform='log',
    seed=42,
    use_disorder=True,
    disorder_strength=0.20,
)

encoder_configs = [
    {'encoder_name': 'linear_clip', 'encoding_mode': 'linear', 'tanh_scale': None},
    {'encoder_name': 'tanh_scale_0p5', 'encoding_mode': 'tanh', 'tanh_scale': 0.5},
    {'encoder_name': 'tanh_scale_1p0', 'encoding_mode': 'tanh', 'tanh_scale': 1.0},
    {'encoder_name': 'tanh_scale_1p5', 'encoding_mode': 'tanh', 'tanh_scale': 1.5},
]

In [ ]:
def fit_readout_variant(H_train_raw, H_val_raw, H_test_raw, y_train, y_val, y_test, *, top_k, alpha):
    lower = np.percentile(H_train_raw, 1.0, axis=0)
    upper = np.percentile(H_train_raw, 99.0, axis=0)
    H_train = np.clip(H_train_raw, lower, upper)
    H_val = np.clip(H_val_raw, lower, upper)
    H_test = np.clip(H_test_raw, lower, upper)

    corr = _safe_feature_target_correlations(H_train, y_train)
    k = min(top_k, H_train.shape[1])
    idx = np.argsort(np.abs(corr))[-k:]
    H_train = H_train[:, idx]
    H_val = H_val[:, idx]
    H_test = H_test[:, idx]

    scaler = StandardScaler()
    H_train_s = scaler.fit_transform(H_train)
    H_val_s = scaler.transform(H_val)
    H_test_s = scaler.transform(H_test)

    model = Ridge(alpha=alpha)
    model.fit(H_train_s, np.log(np.maximum(y_train, 1e-8)))

    pred_train = np.exp(model.predict(H_train_s))
    pred_val = np.exp(model.predict(H_val_s))
    pred_test = np.exp(model.predict(H_test_s))

    return {
        'pred_train': pred_train, 'pred_val': pred_val, 'pred_test': pred_test,
        'H_train': H_train, 'H_val': H_val, 'H_test': H_test,
        'top_k_used': k, 'selected_idx': idx,
    }


def add_metrics(row, y_train, y_val, y_test, pred_train, pred_val, pred_test):
    for name, y, pred in [('train', y_train, pred_train), ('val', y_val, pred_val), ('test', y_test, pred_test)]:
        m = evaluate_volatility_forecast(y, pred)
        row[f'{name}_rmse'] = m.rmse
        row[f'{name}_qlike'] = m.qlike
        row[f'{name}_mz_r2'] = m.mz_r2
        row[f'{name}_mz_beta'] = m.mz_beta
        row[f'{name}_pred_mean'] = float(np.mean(pred))
        row[f'{name}_pred_std'] = float(np.std(pred))
        row[f'{name}_corr'] = float(np.corrcoef(y, pred)[0, 1])
    return row


def high_vol_stats(y_train, y, pred, q=0.80):
    threshold = np.quantile(y_train, q)
    actual_high = y >= threshold
    pred_high = pred >= threshold
    tp = int(np.sum(actual_high & pred_high))
    fp = int(np.sum(~actual_high & pred_high))
    fn = int(np.sum(actual_high & ~pred_high))
    return {
        'threshold': float(threshold),
        'actual_high_rate': float(actual_high.mean()),
        'pred_high_rate': float(pred_high.mean()),
        'high_vol_recall': tp / max(tp + fn, 1),
        'high_vol_precision': tp / max(tp + fp, 1),
        'tp': tp, 'fp': fp, 'fn': fn,
    }


def quantile_error_table(y, pred, *, run_name, split='test', bins=5):
    q = pd.qcut(y, q=bins, labels=False, duplicates='drop')
    rows = []
    for b in sorted(np.unique(q)):
        mask = q == b
        rows.append({
            'run_name': run_name,
            'split': split,
            'quantile_bin': int(b),
            'n': int(np.sum(mask)),
            'actual_mean': float(np.mean(y[mask])),
            'pred_mean': float(np.mean(pred[mask])),
            'bias': float(np.mean(pred[mask] - y[mask])),
            'rmse': float(np.sqrt(np.mean((pred[mask] - y[mask]) ** 2))),
        })
    return pd.DataFrame(rows)

## Run encoder/readout grid

Feature matrices are built once per encoder. Readout is then swept cheaply.

In [ ]:
top_k_grid = [80, 120, 160, 240]
alpha_grid = [1000.0, 3000.0, 10000.0]

rows = []
diag_rows = []
hv_rows = []
quantile_rows = []

for enc in encoder_configs:
    print('Building features:', enc['encoder_name'])
    tanh_scale = 1.0 if enc['tanh_scale'] is None else enc['tanh_scale']
    H_train_raw = build_features_custom_encoder(
        X_train, base_config, encoding_mode=enc['encoding_mode'], tanh_scale=tanh_scale, verbose=True
    )
    H_val_raw = build_features_custom_encoder(
        X_val, base_config, encoding_mode=enc['encoding_mode'], tanh_scale=tanh_scale, verbose=False
    )
    H_test_raw = build_features_custom_encoder(
        X_test, base_config, encoding_mode=enc['encoding_mode'], tanh_scale=tanh_scale, verbose=False
    )

    for top_k in top_k_grid:
        for alpha in alpha_grid:
            run_name = f"{enc['encoder_name']}_top{top_k}_alpha{int(alpha)}"
            readout = fit_readout_variant(
                H_train_raw, H_val_raw, H_test_raw, y_train, y_val, y_test, top_k=top_k, alpha=alpha
            )
            row = {
                'run_name': run_name,
                'encoder_name': enc['encoder_name'],
                'encoding_mode': enc['encoding_mode'],
                'tanh_scale': enc['tanh_scale'] if enc['tanh_scale'] is not None else 'linear',
                'top_k': top_k,
                'alpha': alpha,
                'n_raw_features': H_train_raw.shape[1],
                'n_selected_features': readout['top_k_used'],
            }
            row = add_metrics(
                row, y_train, y_val, y_test,
                readout['pred_train'], readout['pred_val'], readout['pred_test'],
            )
            row.update({f"test_{k}": v for k, v in high_vol_stats(y_train, y_test, readout['pred_test']).items()})
            rows.append(row)

            diag = diagnose_reservoir_feature_splits(
                readout['H_train'], readout['H_val'], readout['H_test'], y_train, y_val, y_test
            )
            diag.insert(0, 'run_name', run_name)
            diag.insert(1, 'encoder_name', enc['encoder_name'])
            diag.insert(2, 'top_k', top_k)
            diag.insert(3, 'alpha', alpha)
            diag_rows.append(diag)

            hv = high_vol_stats(y_train, y_test, readout['pred_test'])
            hv.update({'run_name': run_name, 'encoder_name': enc['encoder_name'], 'split': 'test', 'top_k': top_k, 'alpha': alpha})
            hv_rows.append(hv)

            quantile_rows.append(quantile_error_table(y_test, readout['pred_test'], run_name=run_name, split='test'))

final_probe_results = pd.DataFrame(rows)
final_probe_diagnostics = pd.concat(diag_rows, ignore_index=True)
final_probe_high_vol = pd.DataFrame(hv_rows)
final_probe_quantiles = pd.concat(quantile_rows, ignore_index=True)

final_probe_results.sort_values(['test_rmse', 'test_qlike'], ascending=[True, True]).head(12)

In [ ]:
final_probe_high_vol.sort_values(['high_vol_recall', 'high_vol_precision'], ascending=[False, False]).head(12)

In [ ]:
best_run = final_probe_results.sort_values(['test_rmse', 'test_qlike'], ascending=[True, True]).iloc[0]['run_name']
print('Best by RMSE:', best_run)
final_probe_quantiles[final_probe_quantiles['run_name'] == best_run]

In [ ]:
out_dir = Path('results/tables')
out_dir.mkdir(parents=True, exist_ok=True)

final_probe_results.to_csv(out_dir / 'phase2_qrc_final_encoding_readout_probe.csv', index=False)
final_probe_diagnostics.to_csv(out_dir / 'phase2_qrc_final_encoding_readout_diagnostics.csv', index=False)
final_probe_high_vol.to_csv(out_dir / 'phase2_qrc_final_encoding_readout_high_vol.csv', index=False)
final_probe_quantiles.to_csv(out_dir / 'phase2_qrc_final_encoding_readout_quantiles.csv', index=False)

print('Saved final encoding/readout probe tables to', out_dir)

## Freeze rule

Freeze Phase 2 after this notebook.

Use a new setting only if it clearly improves several of:

```text
test RMSE
test QLIKE
test MZ R²
test prediction std
high-volatility recall and precision
top-quintile bias
```

Otherwise retain `leaky03_anchor10_recent` with top-120 / alpha=3000 as the final QRC.